# SignVoice — reproducible GPU training
Continuous Indian Sign Language → English text → speech.

**Status:** This notebook contains executable training code, not recorded training results. Dataset access must be verified first. Do not present synthetic smoke checks as research results.

Choose **Runtime → Change runtime type → T4 GPU**. Upload the project bundle when prompted (or put the project in `/content/SignVoice`). Training uses an isolated Python 3.11 environment because Colab's default Python may differ. Allow 15–30 minutes for initial installs. Start with 100 valid examples, then 1,000–3,000 once the complete pipeline succeeds.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os, sys, subprocess, zipfile

drive.mount('/content/drive')
PROJECT = Path('/content/SignVoice')
DRIVE = Path('/content/drive/MyDrive/SignVoice')
DRIVE.mkdir(parents=True, exist_ok=True)
if not (PROJECT / 'src/train.py').exists():
    print('Upload SignVoice-project.zip from the generated project artifacts.')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    if len(archives) != 1: raise ValueError('Upload exactly one project ZIP')
    with zipfile.ZipFile(archives[0]) as z:
        for member in z.infolist():
            dest = (Path('/content') / member.filename).resolve()
            if not dest.is_relative_to(Path('/content')): raise ValueError('Unsafe ZIP path')
            if (member.external_attr >> 16) & 0o170000 == 0o120000: raise ValueError('Symlinks not allowed')
        z.extractall('/content')
assert (PROJECT / 'src/train.py').exists(), 'Bundle must contain SignVoice/src/train.py'
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.12.23'], check=True)
subprocess.run(['uv', 'python', 'install', '3.11.17'], check=True)
if not (PROJECT / '.venv/bin/python').exists():
    subprocess.run(['uv', 'venv', '--python', '3.11.17', '.venv'], check=True)
PY = str(PROJECT / '.venv/bin/python')
subprocess.run(['uv', 'pip', 'install', '--python', PY, '-r', 'requirements.txt'], check=True)
def run(*args):
    return subprocess.run([PY, *map(str,args)], cwd=PROJECT, check=True)
run('-c', "import torch,mediapipe; print('PyTorch',torch.__version__,'CUDA',torch.cuda.is_available()); print('GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'); assert torch.cuda.is_available(), 'Select a GPU runtime before training'")


## 1. Dataset access, license and actual file inventory
Inspect both Hugging Face candidates before choosing files. ISLTranslate's official GitHub README describes English-aligned raw videos and MediaPipe archives at iSign, under CC BY-NC 4.0; current HF access and license must also be checked. GloSL-ISL and Kaggle fallback were not accessible from the development environment. Never assume they have videos or a matching landmark format.

If HF requires acceptance, accept the dataset conditions on its page first and use `huggingface_hub.login()` through Colab's secure token prompt if needed. Never place tokens in this notebook or Git. Only use data you are authorized to access.


In [ ]:
run('scripts/inspect_datasets.py')
import json
inventory = json.loads(Path('reports/dataset_inventory.json').read_text())
for item in inventory['datasets']:
    print(item.get('repo'), 'license:', item.get('license'), 'error:', item.get('error'))
    print(item.get('readme', '')[:16000])


## 2. Select real raw-video assets
**Intervention required:** From the printed inventory, select exact files containing raw ISL videos and aligned English CSV annotations. Prefer a small sample video archive. Set `HF_FILES`, `CSV_PATH`, and `VIDEO_DIR` below after inspection. Their names cannot be guessed in advance while dataset access is blocked.

This pipeline deliberately re-extracts raw videos. Precomputed poses are not accepted merely because their dimensions match. If only incompatible pose files are available, stop: webcam/upload compatibility has not been established. Do not substitute random features.


In [ ]:
REPO = 'Exploration-Lab/iSign'
HF_FILES = []  # Exact existing filenames from inventory; no guessed archive names.
MAX_DOWNLOAD_GB = 8
if not HF_FILES:
    raise ValueError('Set HF_FILES from the actual inventory. Select raw video and annotation assets, not pose-only files.')
from importlib.util import spec_from_file_location, module_from_spec
# Use the pinned environment for the downloader.
run('scripts/download_data.py', '--repo', REPO, '--files', *HF_FILES,
    '--output', 'data/downloads', '--max-gb', MAX_DOWNLOAD_GB)
for file in HF_FILES:
    path = Path('data/downloads') / file
    if zipfile.is_zipfile(path) or str(path).endswith(('.tar', '.tar.gz', '.tgz')):
        # Distinct destination per archive, no silent overwrites on reruns.
        destination = Path('data/raw') / path.name.replace('.', '_')
        if not destination.exists(): run('scripts/extract_archive.py', path, destination)
print('CSV files:', list(Path('data').rglob('*.csv'))[:30])
print('Video examples:', [str(p) for p in Path('data').rglob('*') if p.suffix.lower() in {'.mp4','.avi','.mov'}][:10])


## 3. Inspect annotation alignment and prepare the first 100 examples
Choose actual paths printed above. The supported verified ISLTranslate annotation schema is `uid,text`; inspect any other version and create an explicit, reviewed mapping before using it. UID must match the video stem exactly. Source grouping expects a final `-segmentNumber`, as documented in ISLTranslate. No filename is used as a prediction.

The split joins source-video groups and repeated normalized text into components before assigning approximately 80/10/10. This prevents source and text overlap; it does not establish signer independence. If official splits become available, import them and audit overlap rather than claiming this derived split is official.


In [ ]:
CSV_PATH = ''  # Set actual English annotation CSV path.
VIDEO_DIR = '' # Set actual extracted raw-video directory.
LIMIT = 100   # After first success, increase to 1000, then 3000 if feasible.
if not CSV_PATH or not VIDEO_DIR: raise ValueError('Set real paths from the inventory/extraction cell')
run('-c', "import pandas as pd,sys; d=pd.read_csv(sys.argv[1]); print(d.columns.tolist()); print(d.head().to_string()); print('annotations:',len(d)); assert {'uid','text'} <= set(d.columns)", CSV_PATH)
run('-m', 'src.prepare', '--csv', CSV_PATH, '--videos', VIDEO_DIR,
    '--output', 'data/prepared', '--limit', LIMIT)
audit = json.loads(Path('data/prepared/audit.json').read_text())
print(json.dumps(audit, indent=2)[:12000])
assert audit['aligned'] >= 100


## 4. Engineering smoke check and tests
This step uses temporary synthetic fixtures. It is not real-data training or a dataset evaluation. The app refuses `models/smoke.pt`.

In [ ]:
run('-m','scripts.smoke')
run('-m','pytest','-q')


## 5. Baseline experiment
The closed-set BiLSTM classifier is trained only if repeated whole-sentence classes have covered validation and test examples. Its separate source-group split permits class text overlap and reports held-out coverage. If ineligible, the main evaluation still computes a most-frequent-training-sentence baseline. Do not compare classifier accuracy directly with BLEU or describe classification as unrestricted translation.


In [ ]:
run('-m','src.baseline','--data','data/prepared','--output',DRIVE/'baseline','--epochs','5')

## 6. First sequence-to-text run
A small two-epoch run verifies real-data training, best-checkpoint saving, and autoregressive test inference. Token vocabulary is fitted on training text only. AdamW, ignored padding, packed lengths, masked attention, clipping, validation scheduling, and early stopping are implemented in `src/train.py`.


In [ ]:
MODEL_DIR = DRIVE / 'models_100'
run('-m','src.train','--data','data/prepared','--output',MODEL_DIR,'--epochs','2','--batch-size','16')
run('-m','src.evaluate','--data','data/prepared','--checkpoint',MODEL_DIR/'best.pt','--output',DRIVE/'evaluation_100.json')


## 7. Scale only after successful initial inference
Change `LIMIT` to 1000–3000, rerun preparation, then set a **new** model directory and train 20–40 epochs. A changed dataset requires a fresh training run. A tiny initial dataset is for debugging, not strong accuracy claims.

To resume the same data/model, leave `MODEL_DIR` unchanged and use `--resume MODEL_DIR/last.pt`, with total `--epochs` greater than the completed epoch. Keep both best.pt and last.pt. Each checkpoint includes optimizer/scheduler state, RNG state, vocabulary, and preprocessing metadata. Checkpoints are written directly to Drive; copy prepared features to Drive if you need to survive runtime deletion.


In [ ]:
# Run after increasing LIMIT and rerunning preparation. Do not silently mix subset results.
# MODEL_DIR = DRIVE / 'models_1000'
# run('-m','src.train','--data','data/prepared','--output',MODEL_DIR,'--epochs','30','--batch-size','16')
# Resume example, only for the SAME prepared data and hidden dimension:
# run('-m','src.train','--data','data/prepared','--output',MODEL_DIR,'--epochs','40','--resume',MODEL_DIR/'last.pt')


## 8. Held-out evaluation, graphs and sample predictions
Test BLEU/WER/exact match are generated from autoregressive predictions, never teacher-forced argmax. The test set must remain unused for model selection. WER can exceed 1; lower is better. BLEU is 0–100 and exact match is 0–1.

In [ ]:
run('-m','src.evaluate','--data','data/prepared','--checkpoint',MODEL_DIR/'best.pt','--output',DRIVE/'evaluation.json')
from IPython.display import display, Image
results = json.loads((DRIVE/'evaluation.json').read_text())
print({k:v for k,v in results.items() if k != 'predictions'})
for row in results['predictions'][:20]: print(row)
display(Image(filename=str(MODEL_DIR/'learning_curves.png')))
import shutil
shutil.copy('data/prepared/audit.json', DRIVE/'data_audit.json')
shutil.copy('reports/dataset_inventory.json', DRIVE/'dataset_inventory.json')
shutil.copy('data/downloads/download_provenance.json', DRIVE/'download_provenance.json')


## 9. Actual raw-video inference
Select a real held-out raw video. Never use a training example to advertise test accuracy. gTTS uses an external service and needs Internet; recognition still returns text if speech is unavailable.


In [ ]:
VIDEO_TEST = '' # Actual held-out raw video path from the test split.
if not VIDEO_TEST: raise ValueError('Set VIDEO_TEST to a real held-out signing video')
run('-c', "from src.inference import Translator; import sys; print(Translator(sys.argv[1]).translate(sys.argv[2]))", MODEL_DIR/'best.pt', VIDEO_TEST)


## 10. Export to the local application
Download the real trained checkpoint and evaluation report. Put them in the project as `models/best.pt` and `reports/evaluation.json`. Start the app using `.venv/bin/python app.py`. Test upload first, then webcam recording using the same signing sequence. The server-side recording pipeline is shared; browser camera permissions require a real browser/device test.


In [ ]:
files.download(str(MODEL_DIR/'best.pt'))
files.download(str(DRIVE/'evaluation.json'))


## Optional Colab-hosted UI
The following cell starts a **public Gradio share link** while the runtime is active. Run it only if you want this temporary public deployment. Do not upload sensitive videos. Local execution is preferable for a viva demonstration. The code itself does not auto-publish the application.


In [ ]:
# Optional: uncomment to start a public demo in Colab.
# import shutil
# shutil.copy(MODEL_DIR/'best.pt','models/best.pt')
# shutil.copy(DRIVE/'evaluation.json','reports/evaluation.json')
# run('-c', 'from app import build_app; build_app().queue().launch(share=True)')


## Submission checklist
- Keep dataset attribution and license, exact revision, split audit and configuration.
- Include real training curves, BLEU/WER/exact match and reference/prediction comparisons.
- Include failure cases and signer/domain limitations; no fabricated accuracy.
- Capture actual upload and webcam screenshots after real inference.
- Synthetic smoke outputs and an untrained UI are not proof of working ISL translation.
